# 5G Network Intelligence - Temporal Dilated Convolution

## Hypothesis and prior evidence

A nonlinear MLP improves the matched linear reference, but needs hurdle blending for better MAE. Test whether temporal weight sharing and dilated convolution learn useful local/multiscale structure with a compact parameter budget. A/B inputs, causal seasonal initialization, loss and optimizer policy match earlier neural studies; improvement is not assumed. Architecture inspiration: [Bai et al., 2018](https://arxiv.org/abs/1803.01271).

## Temporal protocol

Fit/stop/score cutoffs remain 335/383/443 and 395/443/503. Stopping chooses epochs; scoring cannot tune the architecture. Retain every original beam and eligible per-horizon scoring origin. These reused windows cannot prove independent generalization. Validation 504-671, original holdout and production are excluded. No extra dataset is added.


In [ ]:
RUN_TRAINING = False
SOURCE_COMMIT = '5b6c2c87d2a03ed095ac7efd575c31612430fea1'


## Configuration

Four hosted GPU fits: A/B on two folds, one seed. All beams participate in fitting; no beam identity or summary is a numerical model input.


In [ ]:
PROTOCOL = {'protocol': 'temporal-residual-tcn-development-v1',
 'variants': ['A', 'B'],
 'seed': 17,
 'horizons': [1, 6, 24],
 'folds': [{'name': 'inner1', 'fit_end': 335, 'stop_end': 383, 'score_end': 443},
           {'name': 'inner2', 'fit_end': 395, 'stop_end': 443, 'score_end': 503}],
 'dropout': 0.1,
 'learning_rate': 0.001,
 'weight_decay': 0.01,
 'batch_size': 1024,
 'max_epochs': 30,
 'early_stopping_patience': 5,
 'gradient_clip': 1.0,
 'training_stride': 1,
 'beam_population': 'all_original_beams',
 'preprocessing': 'global_per_KPI_fit_prefix_only_with_missing_masks',
 'initialization': 'causal_target_seasonal_mean_7d_plus_zero_residual_head',
 'phase_inputs': 4,
 'validation_504_671': 'forbidden',
 'original_final_holdout': 'forbidden',
 'production_change': 'forbidden',
 'channels': 32,
 'levels': 7,
 'architecture': 'left_only_dilated_residual_convolution_per_time_LayerNorm',
 'receptive_field': 255}


## Preprocessing and architecture

Fit global per-KPI preprocessing only on each fit prefix. Keep physical KPI missing masks and phase4. Center physical history by the last observed-window value, project to 32 channels, and use seven residual blocks with kernel size 3 and dilations 1/2/4/8/16/32/64. Left padding only; LayerNorm is per time across channels. The nominal receptive field is 255, covering the available 168-hour window. A zero-initialized head adds a residual to the causal seven-day seasonal target mean. Weights are shared across beams; no fitted beam summaries or neighbouring beam inputs. This is a compact adaptation, not a reproduction of the original TCN paper. Full source: `src/models/temporal_tcn.py`.


## Hosted source and canonical inputs

Attach the completed original hurdle development output for shared code and canonical panel, plus the small MIT temporal-TCN source dataset. Only the new model, trainer and protocol are copied from that source attachment.


In [ ]:
from pathlib import Path
import os, sys, shutil, subprocess, tarfile
if RUN_TRAINING:
    if not Path('/kaggle/working').is_dir():
        raise RuntimeError('Hosted Kaggle required')
    roots = [p.parent.parent for p in Path('/kaggle/input').rglob('train_hurdle_research.py') if (p.parent.parent/'data/processed/panel.npz').exists()]
    if len(roots) != 1:
        raise RuntimeError('Attach exactly one canonical source output')
    PROJECT = Path('/kaggle/working/temporal_tcn')
    PROJECT.mkdir(exist_ok=True)
    for folder in ('src','scripts','configs','data/processed'):
        shutil.copytree(roots[0]/folder,PROJECT/folder,dirs_exist_ok=True,ignore=shutil.ignore_patterns('__pycache__'))
    os.chdir(PROJECT)
    sys.path.insert(0,str(PROJECT))
    os.environ['SOURCE_GIT_COMMIT'] = SOURCE_COMMIT
    Path('reports').mkdir(exist_ok=True)
    patches = list(Path('/kaggle/input').rglob('train_temporal_tcn.py'))
    if len(patches) != 1:
        raise RuntimeError('Attach exactly one temporal-TCN source')
    patch = patches[0].parent
    for filename,folder in (('train_temporal_mlp.py','scripts'),('train_temporal_tcn.py','scripts'),('temporal_tcn.py','src/models'),('temporal_tcn.yaml','configs')):
        shutil.copyfile(patch/filename,Path(folder)/filename)


## GPU verification

Use the hosted image's PyTorch installation. A missing GPU stops fitting; there is no local or CPU training fallback.


In [ ]:
if RUN_TRAINING:
    import torch
    print('PyTorch:',torch.__version__)
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA GPU is required')
    subprocess.run(['nvidia-smi'],check=True)


## Training and debugging

The shared masked-L1 trainer writes best/last checkpoint, optimizer/RNG state and per-epoch history. AdamW weight decay is fixed at its 0.01 default. The stopping score is joint scaled L1; scoring reports original-unit MAE after nonnegative clipping. Failures preserve partial artifacts and a bounded progress log.


In [ ]:
if RUN_TRAINING:
    with Path('reports/temporal_tcn_training.log').open('w') as log:
        process = subprocess.Popen([sys.executable,'-u','-m','scripts.train_temporal_tcn'],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout:
            print(line,end='',flush=True)
            log.write(line)
            log.flush()
        status = process.wait()
    with tarfile.open('/kaggle/working/temporal_tcn_results.tar.gz','w:gz') as archive:
        for folder in ('reports','artifacts/temporal_tcn'):
            if Path(folder).exists():
                archive.add(folder,arcname=folder)
    if status:
        raise RuntimeError('Training failed; inspect saved log and checkpoints')
else:
    print('Review mode: no fitting, backward or optimizer steps.')


## Scoring, negative results and limitations

Every horizon has its own complete scoring origins, so short horizons are not restricted by +24h label availability. Report MAE in original units, WAPE as a ratio, coverage and zero false-positive rate. One seed and reused development folds cannot establish generalization. No production promotion or holdout replay.


In [ ]:
paths = [Path('reports/temporal_tcn_development.csv'),Path('../reports/temporal_tcn_development.csv')]
if Path('/kaggle/input').is_dir():
    paths += list(Path('/kaggle/input').rglob('temporal_tcn_development.csv'))
path = next((p for p in paths if p.exists()),None)
if path:
    import pandas as pd
    from IPython.display import display
    display(pd.read_csv(path))
else:
    print('No completed scores attached; no improvement is assumed.')


## Completed v1 result and stopping-cap limitation

The queued Kaggle v1 eventually completed four T4 runs; no Colab fitting was
verified. TCN lowers MAE versus Linear in all 12 comparisons and versus MLP in
10/12. TCN-B improves versus MLP-B in all six cases (0.156-0.757%), but TCN is
better than hurdle in only 5/12 cases and still loses every 1h comparison.

Inner1 B reached epoch 30 with only three stale epochs, below patience five.
Its historical budget_exhausted=false flag covers wall-time exhaustion only.
This is an epoch-capped run, not completed early stopping. Keep original files
immutable and do not derive fixed-refit epochs from it. A continuation requires
a separately declared stopping-only budget policy, not validation tuning.

All 32 files, full scoring metrics/axes and fit-prefix preprocessing passed
verification. Seventy-two history-only predictions agree with cloud output
within 1e-5 original units (maximum 1.80e-7). Source checkpoint epoch/stale
counters are checked separately. These are reused development results, not
independent generalization evidence. No production change or validation/holdout
replay. See reports/temporal_tcn_analysis.md and training_termination.csv.
Default Run All remains review-only; the hosted worker snapshot is unchanged.
